# SoftSEEPS with WeatherBench 2 data

SoftSEEPS needs, for the valid time of a forecast, two climatological fields on the forecast grid: `p1` (probability
of a dry day) and `t2` (light/heavy threshold). This notebook takes them from the public
[WeatherBench 2](https://weatherbench2.readthedocs.io/en/latest/data-guide.html) ERA5 climatology and computes
SoftSEEPS of an IFS HRES forecast against ERA5, globally and for a regional (European) domain.

Requirements: `pip install ".[example]"` from the repository root (xarray, zarr, gcsfs, pandas).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import xarray as xr

import sys
sys.path.insert(0, "..")   # use softseeps from this repository; not needed after `pip install`
from softseeps import soft_seeps

# WeatherBench 2 provides the same data at other resolutions (e.g. 240x121, 64x32), see the data guide.
WB2 = "gs://weatherbench2/datasets"
CLIMATOLOGY = f"{WB2}/era5-hourly-climatology/1990-2019_6h_1440x721.zarr"
ERA5 = f"{WB2}/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr"
HRES = f"{WB2}/hres/2016-2022-0012-1440x721.zarr"

def open_zarr(path):
    return xr.open_zarr(path, storage_options={"token": "anon"}, decode_timedelta=True)

## Forecast, observation and climatology

24 h precipitation over the first forecast day. The climatology is selected for the **day of year and hour of the
valid time**. WeatherBench 2 stores precipitation in metres; everything is converted to mm, the unit of
`dry_threshold` and `tau`.

In [ ]:
init = pd.Timestamp("2020-07-01T00")
valid = init + pd.Timedelta(hours=24)
var = "total_precipitation_24hr"
dry_threshold = 0.25   # mm; WeatherBench 2 uses 0.1 mm for 6 h accumulations

forecast = open_zarr(HRES)[var].sel(time=init, prediction_timedelta=pd.Timedelta(hours=24)) * 1000
observation = open_zarr(ERA5)[var].sel(time=valid) * 1000
clim = open_zarr(CLIMATOLOGY).sel(dayofyear=valid.dayofyear, hour=valid.hour)
p1 = clim[f"{var}_seeps_dry_fraction"]
t2 = clim[f"{var}_seeps_threshold"] * 1000

def to_tensor(x):   # same (latitude, longitude) order for all fields
    x = x.sortby("latitude").sortby("longitude").transpose("latitude", "longitude")
    return torch.tensor(x.values, dtype=torch.float32)

## SoftSEEPS as a loss

`pred` stands in for the output of a model. `soft_seeps` returns the (area-weighted) mean over the grid and is
differentiable with respect to `pred`.

In [3]:
pred = to_tensor(forecast).clamp(min=0).requires_grad_(True)
obs, p1_t, t2_t = to_tensor(observation), to_tensor(p1), to_tensor(t2)
weights = torch.cos(torch.deg2rad(torch.tensor(np.sort(forecast.latitude.values))))[:, None]
print(pred.shape, obs.shape, p1_t.shape, t2_t.shape)

score = soft_seeps(pred, obs, p1_t, t2_t, dry_threshold, tau=1.0, weights=weights)
print(score)
score.backward()
print(pred.grad.shape)

torch.Size([721, 1440]) torch.Size([721, 1440]) torch.Size([721, 1440]) torch.Size([721, 1440])
tensor(0.5837, grad_fn=<DivBackward0>)
torch.Size([721, 1440])


## Regional model: Europe

For a regional model, take the climatology on the model's domain (here cut out of the global field; for another
grid, interpolate it to the model grid). The function is the same.

In [4]:
def europe(x):   # 35-72 N, 25 W-45 E
    x = x.assign_coords(longitude=(x.longitude + 180) % 360 - 180).sortby("longitude")
    return x.sortby("latitude").sel(latitude=slice(35, 72), longitude=slice(-25, 45))

pred = to_tensor(europe(forecast)).clamp(min=0).requires_grad_(True)
obs, p1_t, t2_t = (to_tensor(europe(x)) for x in (observation, p1, t2))
weights = torch.cos(torch.deg2rad(torch.tensor(europe(forecast).latitude.values)))[:, None]
print(pred.shape)

score = soft_seeps(pred, obs, p1_t, t2_t, dry_threshold, tau=1.0, weights=weights)
print(score)
score.backward()

torch.Size([149, 281])
tensor(0.5220, grad_fn=<DivBackward0>)
